# FabriBuddy_Live

Keeps the **Job Monitor** current between scans: every minute it reads what is running
(and what a schedule is about to start) **read-only as the service principal**, and rewrites
only the monitor's pages in `FabriBuddy_DB`. The app picks the change up within half a minute.

Runs for `minutes` and stops; schedule it **hourly** (Deploy_FabriBuddy does, with
`LIVE_MONITOR = True`) and the monitor is live all day. It reads the settings of
**Load_FabriBuddy_E2E** — there is nothing to set here. It needs one scan to have run.

Cost: one small Python session for as long as it runs — all day, when scheduled hourly.

[Documentation](https://zan-tavcar.github.io/msfabric-fabribuddy/job-monitor/#live)

In [ ]:
minutes = 55          # how long this run keeps the monitor live
every_seconds = 60    # how often it reads
sql_database_name = "FabriBuddy_DB"

In [ ]:
# FabriBuddy_DB, before the FabriBuddy package is installed. As whoever runs this notebook.
import json, struct, subprocess, sys, time
import requests

FABRIC_API = "https://api.fabric.microsoft.com/v1"
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(method, path, body=None, wait=True):
    """The Fabric REST API as you; waits for long-running operations."""
    headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
    r = requests.request(method, f"{FABRIC_API}{path}", headers=headers, json=body, timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed ({r.status_code}): {r.text[:400]}")
    if r.status_code == 202 and wait and r.headers.get("Location"):
        location, state = r.headers["Location"], None
        for _ in range(200):
            time.sleep(min(int(r.headers.get("Retry-After", 3) or 3), 10))
            r = requests.get(location, headers=headers, timeout=60)
            state = (r.json() if r.text else {}).get("status")
            if state in ("Succeeded", "Failed", "Undefined"):
                break
        if state != "Succeeded":
            raise RuntimeError(f"{method} {path} did not succeed: {r.text[:400]}")
        result = requests.get(location + "/result", headers=headers, timeout=60)
        return result.json() if result.status_code == 200 and result.text else {}
    return r.json() if r.text else {}


def find_sql_database(name):
    """(item id, server, database) of the SQL database called `name` here, or (None, "", "")."""
    for item in fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases").get("value", []):
        if item["displayName"].lower() == name.lower():
            props = fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases/{item['id']}").get("properties") or {}
            return item["id"], (props.get("serverFqdn") or "").split(",")[0], props.get("databaseName") or ""
    return None, "", ""


def sql_token():
    return notebookutils.credentials.getToken("https://database.windows.net/")


def sql_connect(server, database):
    import pyodbc

    raw = sql_token().encode("utf-16-le")
    return pyodbc.connect(
        f"Driver={{ODBC Driver 18 for SQL Server}};Server={server},1433;Database={database};"
        "Encrypt=yes;TrustServerCertificate=no",
        attrs_before={1256: struct.pack(f"<I{len(raw)}s", len(raw), raw)},
        timeout=30,
    )


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *args])

In [ ]:
# Install the FabriBuddy release that Deploy_FabriBuddy stored in the database.
_, SQL_SERVER, SQL_DATABASE = find_sql_database(sql_database_name)
if not SQL_SERVER:
    raise RuntimeError(f"No SQL database called {sql_database_name!r} here. Run Deploy_FabriBuddy first.")
with sql_connect(SQL_SERVER, SQL_DATABASE) as conn:
    row = conn.cursor().execute("SELECT name, version, data FROM dbo.fabribuddy_package").fetchone()
if row is None:
    raise RuntimeError("The FabriBuddy package isn't in the database. Run Deploy_FabriBuddy first.")
wheel = f"/tmp/{row[0]}"
with open(wheel, "wb") as f:
    f.write(bytes(row[2]))
pip_install(wheel)
print(f"FabriBuddy {row[1]}")

In [ ]:
# The settings live in Load_FabriBuddy_E2E's parameters; the secret in Azure Key Vault.
import base64

pipeline = next((i for i in fabric("GET", f"/workspaces/{WORKSPACE_ID}/items?type=DataPipeline").get("value", [])
                 if i["displayName"] == "Load_FabriBuddy_E2E"), None)
if pipeline is None:
    raise RuntimeError("Load_FabriBuddy_E2E isn't in this workspace. Run Deploy_FabriBuddy first.")
definition = fabric("POST", f"/workspaces/{WORKSPACE_ID}/items/{pipeline['id']}/getDefinition")
content = next(json.loads(base64.b64decode(p["payload"])) for p in definition["definition"]["parts"]
               if p["path"] == "pipeline-content.json")
settings = {k: v.get("defaultValue") for k, v in content["properties"].get("parameters", {}).items()}
if settings.get("demo"):
    raise RuntimeError("Load_FabriBuddy_E2E runs the demo: there is nothing live to read.")
missing = [n for n in ("key_vault_url", "sp_tenant_id", "sp_client_id") if not settings.get(n)]
if missing:
    raise ValueError(f"Set {', '.join(missing)} in the parameters of Load_FabriBuddy_E2E.")
client_secret = notebookutils.credentials.getSecret(settings["key_vault_url"], settings.get("sp_secret_name") or "fabribuddy-sp-secret")

In [ ]:
# Live: read what is running, rewrite the monitor's pages, every `every_seconds`.
from fabribuddy_intenant import live

result = live(
    tenant_id=settings["sp_tenant_id"],
    client_id=settings["sp_client_id"],
    client_secret=client_secret,
    sql_server=SQL_SERVER,
    sql_database=SQL_DATABASE,
    sql_token=sql_token,   # a function: the token is renewed when it expires
    minutes=minutes,
    every_seconds=every_seconds,
)
print(json.dumps(result, indent=2))